In [ ]:
import json
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()




--- Turn 1 ---
Claude is calling: get_weather({'city': 'Tokyo'})
Tool returned: {'city': 'Tokyo', 'unit': 'fahrenheit', 'temp': 75, 'condition': 'sunny'}

--- Turn 2 ---
Final answer: The weather in Tokyo is currently **sunny** with a temperature of **75°F** (approximately 24°C).

--- Turn 1 ---
Claude is calling: get_weather({'city': 'San Francisco'})
Tool returned: {'city': 'San Francisco', 'unit': 'fahrenheit', 'temp': 62, 'condition': 'foggy'}
Claude is calling: get_weather({'city': 'London'})
Tool returned: {'city': 'London', 'unit': 'fahrenheit', 'temp': 55, 'condition': 'rainy'}

--- Turn 2 ---
Final answer: Here's the weather comparison:

**San Francisco:**
- Temperature: 62°F
- Condition: Foggy

**London:**
- Temperature: 55°F
- Condition: Rainy

**Umbrella Recommendation:**

- **San Francisco:** You likely don't need an umbrella. It's just foggy, not raining, though it's a bit cool at 62°F, so bring a light jacket.

- **London:** You should definitely bring an umbrella! It's

"Here's the weather comparison:\n\n**San Francisco:**\n- Temperature: 62°F\n- Condition: Foggy\n\n**London:**\n- Temperature: 55°F\n- Condition: Rainy\n\n**Umbrella Recommendation:**\n\n- **San Francisco:** You likely don't need an umbrella. It's just foggy, not raining, though it's a bit cool at 62°F, so bring a light jacket.\n\n- **London:** You should definitely bring an umbrella! It's rainy and cooler at 55°F, so also wear a warm jacket or coat along with the umbrella.\n\nIf you're traveling between both cities, an umbrella would be essential for London, and a light jacket would be helpful for both locations."

: 

In [ ]:

# Step 1: Define the tool
weather_tool = {
    "name": "get_weather",
    "description": "Get the current weather in a given city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city name, e.g. 'San Francisco'"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "default": "fahrenheit"
            }
        },
        "required": ["city"]
    }
}


# Step 2: Implement the fake tool
def get_weather(city: str, unit: str = "fahrenheit") -> dict:
    """Fake weather data for testing."""
    fake_data = {
        "San Francisco": {"temp": 62, "condition": "foggy"},
        "Tokyo": {"temp": 75, "condition": "sunny"},
        "London": {"temp": 55, "condition": "rainy"},
    }
    data = fake_data.get(city, {"temp": 70, "condition": "unknown"})
    if unit == "celsius":
        data["temp"] = round((data["temp"] - 32) * 5 / 9)
    return {"city": city, "unit": unit, **data}


# Step 3: The conversation loop
def agent_query(user_message: str, max_turns: int = 5) -> str:
    messages = [{"role": "user", "content": user_message}]
    
    for turn in range(max_turns):
        print(f"\n--- Turn {turn + 1} ---")
        
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=[weather_tool],
            messages=messages,
        )
        
        # If the model is done (no more tool calls), return its final text
        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if b.type == "text")
            print(f"Final answer: {final}")
            return final
        
        # The model wants to call a tool. Append its response to the conversation.
        messages.append({"role": "assistant", "content": response.content})
        
        # Process every tool_use block
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"Claude is calling: {block.name}({block.input})")
                if block.name == "get_weather":
                    result = get_weather(**block.input)
                    print(f"Tool returned: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result),
                    })
        
        # Send tool results back to the model
        messages.append({"role": "user", "content": tool_results})
    
    raise RuntimeError(f"Agent did not finish in {max_turns} turns")


In [ ]:


# Try it
agent_query("What's the weather in Tokyo?")
agent_query("Compare the weather in San Francisco and London. Should I bring an umbrella?")

End-of-day reflection:

When does the loop terminate?
What if the model never calls a tool — does the code handle that?
What if the model makes up a city name not in your fake data?
Where in this code would production-quality error handling go?